# 图2子图导出：总体统计与代表性案例

本 notebook 只读取已经保存的多城市检索结果和轴测图，不重新计算指标，也不访问数据库。运行后输出四张可在排版软件中独立组合的 PNG：

1. 50 个正式查询的屋顶增益—总体形态代价散点图；
2. 15号中值状态案例的检索对照；
3. 16号大坡度案例的检索对照；
4. 13号高离散度案例的检索对照。

图片只写入文件，不在单元格输出区显示。子图总标题和最终编号留待排版时添加。

In [1]:
from __future__ import annotations

from pathlib import Path
import os
import sys

import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
import numpy as np
import pandas as pd
from PIL import Image, ImageChops, ImageDraw, ImageFont

def find_notebook_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'multicity_visualization.py').is_file():
        return current
    for parent in (current, *current.parents):
        candidate = parent / 'casebase' / 'lod2_roof_retrieval_experiment'
        if (candidate / 'multicity_visualization.py').is_file():
            return candidate
    raise FileNotFoundError('未找到 lod2_roof_retrieval_experiment 目录。')

NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import multicity_visualization as mviz

FONT_REGULAR_PATH = Path(r'C:\Windows\Fonts\Deng.ttf')
FONT_BOLD_PATH = Path(r'C:\Windows\Fonts\Dengb.ttf')
for font_path in (FONT_REGULAR_PATH, FONT_BOLD_PATH):
    if not font_path.is_file():
        raise FileNotFoundError(f'未找到等线字体：{font_path}')
    fontManager.addfont(str(font_path))
FONT_CN = FontProperties(fname=str(FONT_REGULAR_PATH))
FONT_CN_BOLD = FontProperties(fname=str(FONT_BOLD_PATH))
mpl.rcParams['font.family'] = FONT_CN.get_name()
mpl.rcParams['axes.unicode_minus'] = False


## 1. 文件、案例与版面参数

统计图采用横向画幅；三个案例采用相同尺寸的横向对照带。对照带依次放置查询街区、总体形态检索前5名和屋顶增强检索前5名。

In [2]:
TOP_K = 5
PRIMARY_LAMBDA = 0.35
OUTPUT_DPI = 300
SCATTER_FIGSIZE = (16.0, 5.2)
CASE_TILE_SIZE = (420, 330)

RESULT_DIR = NOTEBOOK_DIR / 'outputs' / 'multicity' / 'global_queries'
RETRIEVAL_PATH = RESULT_DIR / 'global_retrieval_results.csv'
SOURCE_FIGURE_DIR = RESULT_DIR / 'figures' / 'all_queries_ranked_axonometric'
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / 'paper_figure_assets' / 'figure2_panels'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_STYLE = {
    'near_flat': ('平缓屋顶', '#8A8A8A'),
    'typical': ('中值状态', '#4C78A8'),
    'steep': ('大坡度', '#E45756'),
    'high_relief': ('高落差', '#72B7B2'),
    'heterogeneous': ('高离散度', '#B279A2'),
}

CASE_SPECS = [
    {
        'number': 15, 'query_id': 'CZ_PR_000849', 'category': '中值状态',
        'source': '15_typical_CZ_PR_000849_axonometric.png',
        'output': 'b_case_15_CZ_PR_000849.png',
    },
    {
        'number': 16, 'query_id': 'DE_BE_008010', 'category': '大坡度',
        'source': '16_steep_DE_BE_008010_axonometric.png',
        'output': 'c_case_16_DE_BE_008010.png',
    },
    {
        'number': 13, 'query_id': 'DE_BE_002692', 'category': '高离散度',
        'source': '13_heterogeneous_DE_BE_002692_axonometric.png',
        'output': 'd_case_13_DE_BE_002692.png',
    },
]

required_paths = [RETRIEVAL_PATH] + [SOURCE_FIGURE_DIR / item['source'] for item in CASE_SPECS]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('缺少05或04的输出文件：\n' + '\n'.join(missing_paths))

print(f'输出目录：{OUTPUT_DIR}')
print('准备输出：1张统计图和3张案例对照图。')


输出目录：E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels
准备输出：1张统计图和3张案例对照图。


## 2. 总体统计散点图

横轴表示加入屋顶信息后总体形态距离的相对增幅，越靠右表示总体形态代价越大；纵轴表示屋顶距离的相对降幅，越靠上表示屋顶匹配改善越明显。虚线标出50个查询的中位水平，普通查询按五种屋顶情景着色，三个展示案例使用黑色外圈强调。

In [3]:
retrieval = pd.read_csv(RETRIEVAL_PATH, encoding='utf-8-sig')
diagnostics = mviz.build_query_gain_cost(retrieval, top_k=TOP_K).copy()
diagnostics['general_cost_percent'] = np.where(
    diagnostics['m1_general_mean'] > 0,
    100.0 * diagnostics['general_cost'] / diagnostics['m1_general_mean'],
    0.0,
)
diagnostics['roof_gain_percent'] = np.where(
    diagnostics['m1_roof_mean'] > 0,
    100.0 * diagnostics['roof_gain'] / diagnostics['m1_roof_mean'],
    0.0,
)

def save_gain_cost_scatter(frame: pd.DataFrame, output_path: Path) -> Path:
    fig, ax = plt.subplots(figsize=SCATTER_FIGSIZE, facecolor='white')
    for category, (label, color) in CATEGORY_STYLE.items():
        subset = frame[frame['query_category'] == category]
        ax.scatter(
            subset['general_cost_percent'], subset['roof_gain_percent'],
            s=62, color=color, alpha=0.76, edgecolors='white', linewidths=0.7,
            label=label, zorder=3,
        )

    x_median = float(frame['general_cost_percent'].median())
    y_median = float(frame['roof_gain_percent'].median())
    ax.axvline(x_median, color='#777777', linewidth=1.0, linestyle=(0, (4, 4)), zorder=1)
    ax.axhline(y_median, color='#777777', linewidth=1.0, linestyle=(0, (4, 4)), zorder=1)
    ax.text(
        x_median, 1.01, f'总体形态代价中位数 {x_median:.1f}%',
        transform=ax.get_xaxis_transform(), ha='center', va='bottom',
        fontsize=10.5, color='#666666', fontproperties=FONT_CN,
    )
    ax.text(
        1.005, y_median, f'屋顶改善中位数 {y_median:.1f}%',
        transform=ax.get_yaxis_transform(), ha='left', va='center',
        fontsize=10.5, color='#666666', fontproperties=FONT_CN,
    )

    label_offsets = {15: (10, -20), 16: (10, 10), 13: (10, 10)}
    for item in CASE_SPECS:
        row = frame[frame['query_id'].astype(str) == item['query_id']]
        if len(row) != 1:
            raise ValueError(f"查询 {item['query_id']} 在统计表中不是唯一一行。")
        point = row.iloc[0]
        x = float(point['general_cost_percent'])
        y = float(point['roof_gain_percent'])
        ax.scatter(
            [x], [y], s=150, facecolors='none', edgecolors='#222222',
            linewidths=1.8, zorder=5,
        )
        ax.annotate(
            f"{item['number']}号·{item['category']}", (x, y),
            xytext=label_offsets[item['number']], textcoords='offset points',
            fontsize=11.5, color='#222222', fontproperties=FONT_CN_BOLD,
            arrowprops={'arrowstyle': '-', 'color': '#777777', 'lw': 0.8},
            zorder=6,
        )

    x_max = max(55.0, float(frame['general_cost_percent'].max()) * 1.08)
    y_max = max(105.0, float(frame['roof_gain_percent'].max()) * 1.05)
    ax.set_xlim(-2.5, x_max)
    ax.set_ylim(-5.0, y_max)
    ax.set_xlabel('总体形态距离增幅（%）', fontsize=13, labelpad=10, fontproperties=FONT_CN)
    ax.set_ylabel('屋顶距离降幅（%）', fontsize=13, labelpad=10, fontproperties=FONT_CN)
    ax.grid(True, color='#E4E4E4', linewidth=0.7, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#777777')
    ax.spines['bottom'].set_color('#777777')
    ax.tick_params(colors='#555555', labelsize=10.5)
    legend = ax.legend(
        loc='lower right', ncol=1, frameon=True, framealpha=0.94,
        facecolor='white', edgecolor='#DDDDDD', fontsize=10.5,
    )
    for text in legend.get_texts():
        text.set_fontproperties(FONT_CN)
    fig.subplots_adjust(left=0.065, right=0.91, top=0.90, bottom=0.20)
    fig.savefig(output_path, dpi=OUTPUT_DPI, facecolor='white', bbox_inches='tight')
    plt.close(fig)
    return output_path

scatter_path = save_gain_cost_scatter(
    diagnostics, OUTPUT_DIR / 'a_query_gain_cost_scatter.png'
)
print(f'已保存统计图：{scatter_path}')


已保存统计图：E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\a_query_gain_cost_scatter.png


## 3. 三个案例的横向检索对照带

每张图只保留模型、两种方法的分组标签、候选名次和三项核心结果。原图中的英文标题、案例编号、G/R值及面积信息均不保留。所有中文文字均直接调用 Windows 等线字体。

In [4]:
QUERY_BOX = (36, 790, 898, 1470)
M1_BOXES = (
    (988, 407, 1526, 820),
    (1608, 407, 2147, 820),
    (2228, 407, 2768, 820),
    (2849, 407, 3389, 820),
    (3470, 407, 4005, 820),
)
M2_BOXES = tuple((left, 1440, right, 1850) for left, _, right, _ in M1_BOXES)

def trim_and_fit(image: Image.Image, size: tuple[int, int] = CASE_TILE_SIZE) -> Image.Image:
    rgb = image.convert('RGB')
    difference = ImageChops.difference(rgb, Image.new('RGB', rgb.size, 'white'))
    mask = difference.convert('L').point(lambda value: 255 if value > 8 else 0)
    bbox = mask.getbbox()
    if bbox is None:
        raise ValueError('裁切区域中没有检测到模型。')
    model = rgb.crop(bbox)
    model.thumbnail((size[0] - 28, size[1] - 24), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', size, 'white')
    canvas.paste(model, ((size[0] - model.width) // 2, (size[1] - model.height) // 2))
    return canvas

def extract_case_tiles(source_path: Path) -> tuple[Image.Image, list[Image.Image], list[Image.Image]]:
    source = Image.open(source_path).convert('RGB')
    if source.size != (4041, 1911):
        raise ValueError(f'{source_path.name} 尺寸应为4041×1911，实际为{source.size}。')
    query_crop = source.crop(QUERY_BOX)
    ImageDraw.Draw(query_crop).rectangle((658, 460, 733, 680), fill='white')
    query = trim_and_fit(query_crop)
    m1 = [trim_and_fit(source.crop(box)) for box in M1_BOXES]
    m2 = [trim_and_fit(source.crop(box)) for box in M2_BOXES]
    return query, m1, m2

def text_width(draw: ImageDraw.ImageDraw, text: str, font: ImageFont.FreeTypeFont) -> int:
    left, _, right, _ = draw.textbbox((0, 0), text, font=font)
    return right - left

def draw_centered_text(
    draw: ImageDraw.ImageDraw, text: str, center_x: int, y: int,
    font: ImageFont.FreeTypeFont, fill: str,
) -> None:
    draw.text((center_x - text_width(draw, text, font) // 2, y), text, font=font, fill=fill)

def save_case_strip(item: dict, metric_row: pd.Series, output_path: Path) -> Path:
    query, m1, m2 = extract_case_tiles(SOURCE_FIGURE_DIR / item['source'])
    tile_w, tile_h = CASE_TILE_SIZE
    regular_gap = 14
    group_gap = 46
    header_h = 92
    footer_h = 72

    query_x = 0
    m1_start = query_x + tile_w + group_gap
    m1_x = [m1_start + i * (tile_w + regular_gap) for i in range(TOP_K)]
    m2_start = m1_x[-1] + tile_w + group_gap
    m2_x = [m2_start + i * (tile_w + regular_gap) for i in range(TOP_K)]
    canvas_w = m2_x[-1] + tile_w
    canvas_h = header_h + tile_h + footer_h
    canvas = Image.new('RGB', (canvas_w, canvas_h), 'white')
    draw = ImageDraw.Draw(canvas)

    header_font = ImageFont.truetype(str(FONT_BOLD_PATH), 34)
    rank_font = ImageFont.truetype(str(FONT_REGULAR_PATH), 25)
    footer_font = ImageFont.truetype(str(FONT_REGULAR_PATH), 27)
    query_color = '#555555'
    m1_color = '#4C78A8'
    m2_color = '#D66B4F'

    draw_centered_text(draw, '查询街区', query_x + tile_w // 2, 18, header_font, query_color)
    draw_centered_text(
        draw, '总体形态检索', (m1_x[0] + m1_x[-1] + tile_w) // 2,
        18, header_font, m1_color,
    )
    draw_centered_text(
        draw, '屋顶增强检索', (m2_x[0] + m2_x[-1] + tile_w) // 2,
        18, header_font, m2_color,
    )

    canvas.paste(query, (query_x, header_h))
    draw.rectangle(
        (query_x, header_h, query_x + tile_w - 1, header_h + tile_h - 1),
        outline='#A8A8A8', width=2,
    )
    for method_tiles, positions, color in ((m1, m1_x, m1_color), (m2, m2_x, m2_color)):
        for rank, (tile, x) in enumerate(zip(method_tiles, positions), start=1):
            canvas.paste(tile, (x, header_h))
            draw.rectangle(
                (x, header_h, x + tile_w - 1, header_h + tile_h - 1),
                outline=color, width=2,
            )
            rank_text = str(rank)
            draw.ellipse((x + 12, header_h + 12, x + 50, header_h + 50), fill='white', outline=color, width=2)
            draw_centered_text(draw, rank_text, x + 31, header_h + 14, rank_font, color)

    divider_y = header_h + tile_h + 17
    draw.line((0, divider_y, canvas_w, divider_y), fill='#D8D8D8', width=1)
    roof_percent = (
        100.0 * float(metric_row['roof_gain']) / float(metric_row['m1_roof_mean'])
        if float(metric_row['m1_roof_mean']) > 0 else 0.0
    )
    general_percent = (
        100.0 * float(metric_row['general_cost']) / float(metric_row['m1_general_mean'])
        if float(metric_row['m1_general_mean']) > 0 else 0.0
    )
    summary = (
        f"屋顶距离下降 {roof_percent:.1f}%　｜　"
        f"总体形态距离上升 {general_percent:.1f}%　｜　"
        f"前5名更替 {int(metric_row['replacement_count'])} 个"
    )
    draw_centered_text(draw, summary, canvas_w // 2, divider_y + 15, footer_font, '#444444')
    canvas.save(output_path, dpi=(OUTPUT_DPI, OUTPUT_DPI))
    return output_path


In [ ]:
written_paths = [scatter_path]
for item in CASE_SPECS:
    metric_rows = diagnostics[diagnostics['query_id'].astype(str) == item['query_id']]
    if len(metric_rows) != 1:
        raise ValueError(f"查询 {item['query_id']} 在统计表中不是唯一一行。")
    output_path = save_case_strip(
        item, metric_rows.iloc[0], OUTPUT_DIR / item['output']
    )
    written_paths.append(output_path)
    print(f"已保存{item['number']}号案例：{output_path}")

print('\n全部完成，共输出4张子图：')
for path in written_paths:
    print(path)


已保存15号案例：E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\b_case_15_CZ_PR_000849.png
已保存16号案例：E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\c_case_16_DE_BE_008010.png
已保存13号案例：E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\d_case_13_DE_BE_002692.png

全部完成，共输出4张子图：
E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\a_query_gain_cost_scatter.png
E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\b_case_15_CZ_PR_000849.png
E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\c_case_16_DE_BE_008010.png
E:\0_code\block-gml-casebase\casebase\lod2_roof_retrieval_experiment\outputs\paper_figure_assets\figure2_panels\d_case_13

: 

## 4. 排版建议

- 统计散点图适合置于图2上方，横向通栏。
- 三个案例图使用完全一致的尺寸，适合在其下方依次排列。
- 在排版软件中统一添加（a）—（d）编号及案例图题，不需要再次修改本 notebook。
- 如需调整输出比例，只修改 `SCATTER_FIGSIZE` 或 `CASE_TILE_SIZE` 后重新运行即可。